In [1]:
#Use the same ground truth questions:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
# Load the FAQ documents and the search index:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
# Create a lookup table:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc


In [4]:
# First, set up the model clients:
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

load_dotenv()
openai_client = OpenAI()

In [6]:
# Define search tool:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 2.0, "answer": 10.0, "section": 0.2},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [7]:
# Create the runner:

from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [8]:
# The result contains:
# last_message: the final response
# all_messages: the full message history
# cost: the cost of all LLM calls in this run

In [9]:
# Run the runner for 1 ground truth question:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])


In [10]:
# Look at the full message history:
result.all_messages


[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='I just found this course — am I still allowed to join, or is it too late?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"late to join course allowed enroll after start FAQ"}', call_id='call_VifGqqWn7G1DMuUbb8mWRMaN', name='search', type='function_call', id='fc_05806f37f53d0764006a527a57b1c481a3a6edae435ff757a6', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_VifGqqWn7G1DMuUbb8mWRMaN',
  'output': '[\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\n    "answer": "No, you can only get a certificate if you finish the c

In [ ]:
# For this lesson, the search is only the tool calls. We don't need to send the full message history to the judge.
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

In [12]:
# Get tool calls for 1 ground truth question
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"late to join course allowed enroll after start FAQ"}'}]

In [13]:
# Get the original answer:

doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

In [14]:
# Save the A->Q->A' record and the trajectory:

agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

{'question': 'I just found this course — am I still allowed to join, or is it too late?',
 'answer_agent': 'Yes — you can still join.\n\nAccording to the course FAQ, **you can start whenever you want**, and you can also **begin learning and submit homework while the form is open**, even if you didn’t register in advance. Registration is only used to gauge interest before the start date, and it **isn’t checked against a registered list**.\n\nOne caveat: if you want to submit homework, you can only do so **while the homework form is still open** — there are **no late submissions** after it closes.\n\nIf you want, I can also point you to the recommended “start here” materials.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': [{'name': 'search',
   'arguments': '{"query":"late to join course allowed enroll after start FAQ"}'}],
 'cost': Decimal('0.0014670'),
 'document': '74eb249bbf'}

In [15]:
# The answer_agent field is what we evaluate with the LLM judge. 
# The tool_calls field lets the judge see how the agent got there.
# Processing multiple questions:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

In [16]:
# Run it for a small sample in parallel:

from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

  0%|          | 0/50 [00:00<?, ?it/s]

In [17]:
# Turn it into a dataframe:
df_agent = pd.DataFrame(agent_answers)


In [18]:
df_agent.head()

,question,answer_agent,answer_orig,tool_calls,cost,document
0,I just found this course — am I still allowed ...,Yes — you can still join the course.\n\nAccord...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""la...",0.001017,74eb249bbf
1,"If I join now, can I still get a certificate s...",Yes — if you join while the course is still ru...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""ce...",0.00120075,74eb249bbf
2,What do I need to do to qualify for the certif...,"If you’re starting late, you can still join th...","Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""ce...",0.00137775,74eb249bbf
3,Is there a deadline for submitting the project...,"Yes — if you want the certificate, you need to...","Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""de...",0.00120675,74eb249bbf
4,Can I enroll after the course has already star...,Yes — you can still catch up if you enroll aft...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""en...",0.00161025,74eb249bbf


In [19]:
# Calculate the total cost:
df_agent["cost"].sum()



Decimal('0.06710925')

In [20]:
# Save the results
df_agent.to_csv("data/agent-answers.csv", index=False)


In [21]:
## Trajectory judging - the amount of tool calls
# Now define a judge output type with two scores - for the answer and trajectory:

from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [22]:
# Define judge trajectory:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()



In [23]:
# Define the judge function:

import json
from evaluation_utils import calc_total_price, llm_structured_retry

def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [24]:
# Test it on one agent result:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval


AgentEvaluation(answer_reasoning='The agent answer matches the ground truth. It clearly says you can still join the course and includes the key condition for earning a certificate: you must submit your project while submissions are still being accepted. This preserves the essential meaning of the original answer.', answer_score='good', trajectory_reasoning='The single search query is relevant to the question about whether it is too late to join a course. It includes keywords like late, join, and course, and one search is sufficient for this FAQ-style question. No unnecessary or duplicate tool calls were made.', trajectory_score='good')

In [25]:
# Run the judge for all agent answers:

def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [26]:
# Use the same parallel helper:

with ThreadPoolExecutor(max_workers=9) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

  0%|          | 0/50 [00:00<?, ?it/s]

In [27]:
# Split the results:

agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [28]:
# Create a dataframe:

df_agent_eval = pd.DataFrame(agent_evaluations)

In [29]:
# Calculate the judge cost from the token usage:

calc_total_price(usages)

0.05440200000000001

In [30]:
# Check the answer scores:

df_agent_eval["answer_score"].value_counts()

answer_score
good    45
bad      5
Name: count, dtype: int64

In [31]:
# Check the trajectory scores:

df_agent_eval["trajectory_score"].value_counts()

trajectory_score
good    48
bad      2
Name: count, dtype: int64

In [32]:
# Save the judge results:

df_agent_eval.to_csv("data/agent-evaluations.csv", index=False)

In [33]:
df_agent_eval

,question,document,answer_score,answer_reasoning,trajectory_score,trajectory_reasoning
0,I just found this course — am I still allowed ...,74eb249bbf,good,The agent’s answer matches the ground truth. I...,good,The search query was relevant to the question ...
1,"If I join now, can I still get a certificate s...",74eb249bbf,good,The agent’s answer matches the key point of th...,good,The search query was relevant to the question ...
2,What do I need to do to qualify for the certif...,74eb249bbf,good,The agent’s answer includes the core requireme...,good,The single search query was relevant to the qu...
3,Is there a deadline for submitting the project...,74eb249bbf,good,The agent’s answer matches the core ground-tru...,good,The search query was relevant and contained ke...
4,Can I enroll after the course has already star...,74eb249bbf,good,The agent’s answer is broadly consistent with ...,good,The search query was relevant to the question ...
5,Do I actually need a confirmation email to joi...,977bf7786c,good,The agent answer matches the ground truth. It ...,good,The tool call was relevant and included key te...
6,"If I registered for the course, do I have to w...",977bf7786c,good,The agent answer matches the ground truth. It ...,good,The single search query is directly relevant a...
7,"Is there some approved list I need to be on, o...",977bf7786c,good,The agent answer matches the ground truth. It ...,good,The search query was relevant and included key...
8,What’s the point of the registration form if t...,977bf7786c,good,The agent answer matches the ground truth clos...,good,The search query was relevant and included the...
9,Can I start learning and turning in assignment...,977bf7786c,good,The agent answer matches the ground truth well...,good,The search query was relevant and included imp...
